In [41]:
 # 1. imports & setup
from google import genai
from google.genai import types
import json
import re
import os
from dotenv import load_dotenv
from warnings import filterwarnings

filterwarnings("ignore")
load_dotenv()

True

In [42]:
# 2. create client
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL_NAME = os.getenv("GEMINI_MODEL_NAME")

client = genai.Client(api_key = GEMINI_API_KEY)
#print(GEMINI_MODEL_NAME)

In [43]:
# 3. create a prompt template
PROMPT_TEMPLATE = """
You are a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. **Thought**: Analyze the tone, urgency, and patterns.
2. **Action**: Classify if this is likely a scam.
3. **Final Answer**: Output a structured JSON.

Return response only in this JSON format:
```
    {{
    "label": "Scam | Not Scam | Uncertain",
    "reasoning": "Your step-by-step analysis",
    "intent": "Sender's intent behind the message",
    "risk_factors": ["List of red flags like urgency, bad links, etc."]
    }}
```
User Message:
{}
"""

In [44]:
# 4. handle input
# user_input = "Your package delivery failed. Please update your address at [www.bestwebsite.com] to avoid return to sender."
# user_input = "CONGRATULATIONS!! You have won a lottery of 10 million dollars!! Give me your OTP now and take the cash."
# user_input = "Your gmail password is expiring, clicking on the link below to reset the password"
# user_input = "Your ITR Refund is due Enter the account details today to claim"
# user_input = "We found some restricted items in your package. Send your address and CREDIT CARD to resolve this now!"
user_input = "You have an unclaimed policy for $10000, fill the details to claim it, otherwise it will expire in the next 2 hours."

full_prompt = PROMPT_TEMPLATE.format(user_input)
print(full_prompt)


You are a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. **Thought**: Analyze the tone, urgency, and patterns.
2. **Action**: Classify if this is likely a scam.
3. **Final Answer**: Output a structured JSON.

Return response only in this JSON format:
```
    {
    "label": "Scam | Not Scam | Uncertain",
    "reasoning": "Your step-by-step analysis",
    "intent": "Sender's intent behind the message",
    "risk_factors": ["List of red flags like urgency, bad links, etc."]
    }
```
User Message:
You have an unclaimed policy for $10000, fill the details to claim it, otherwise it will expire in the next 2 hours.



In [45]:
# 5. generate llm response
response = client.models.generate_content(
    model = GEMINI_MODEL_NAME,
    contents = full_prompt
)

print(response.text)

1. **Thought**: The message claims an "unclaimed policy" for a significant amount ($10,000). It creates a sense of urgency by stating it will "expire in the next 2 hours." This is a common tactic used by scammers to pressure recipients into acting quickly without thinking. The request to "fill the details" is vague and likely intended to elicit personal or financial information, which is a major red flag. There's no mention of who issued the policy or how it was obtained, which is unusual for legitimate unclaimed funds.

2. **Action**: Based on the urgency, the promise of a large sum of money with little explanation, and the request for personal details under pressure, this message exhibits strong indicators of a scam.

3. **Final Answer**:
```json
    {
    "label": "Scam",
    "reasoning": "The message creates artificial urgency ('expire in the next 2 hours') to pressure the recipient into acting without thinking. It promises a large sum of money ($10,000) without providing clear det

In [46]:
# 6. parse output
def extract_json(text):
    """
    Finds the first '{' and the last '}' in a string 
    and parses it as JSON.
    """
    try:
        # Regex to find the JSON block inside the text
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print(f"JSON Parsing Error: {e}")
    
    # Fallback if parsing fails
    return {
        "label": "Uncertain", 
        "reasoning": "Could not parse AI response",
        "intent": "Unknown",
        "risk_factors": ["Parsing Error"]
    }

parsed_output = extract_json(response.text)
print(parsed_output)

{'label': 'Scam', 'reasoning': "The message creates artificial urgency ('expire in the next 2 hours') to pressure the recipient into acting without thinking. It promises a large sum of money ($10,000) without providing clear details about the source or nature of the 'unclaimed policy'. The request to 'fill the details' is a common phishing tactic to steal personal information.", 'intent': 'To trick the recipient into revealing personal or financial information under the guise of claiming a large sum of money.', 'risk_factors': ['Unsolicited message promising a large sum of money.', 'Creation of artificial urgency.', 'Vague request for personal details.', "Lack of verifiable source or context for the 'unclaimed policy'."]}


In [47]:
# 7. print output
parsed_output

{'label': 'Scam',
 'reasoning': "The message creates artificial urgency ('expire in the next 2 hours') to pressure the recipient into acting without thinking. It promises a large sum of money ($10,000) without providing clear details about the source or nature of the 'unclaimed policy'. The request to 'fill the details' is a common phishing tactic to steal personal information.",
 'intent': 'To trick the recipient into revealing personal or financial information under the guise of claiming a large sum of money.',
 'risk_factors': ['Unsolicited message promising a large sum of money.',
  'Creation of artificial urgency.',
  'Vague request for personal details.',
  "Lack of verifiable source or context for the 'unclaimed policy'."]}

In [48]:
type(parsed_output)

dict

In [49]:
print(parsed_output['label'])
print()
print(parsed_output['reasoning'])

Scam

The message creates artificial urgency ('expire in the next 2 hours') to pressure the recipient into acting without thinking. It promises a large sum of money ($10,000) without providing clear details about the source or nature of the 'unclaimed policy'. The request to 'fill the details' is a common phishing tactic to steal personal information.
